# Data Preparation

This notebook converts the validated raw data into reproducible modelling tables.

The main objectives are to:

1. recreate the documented positive-interaction definition;
2. identify users eligible for chronological evaluation;
3. remove ambiguous users from primary evaluation scoring;
4. create validation and test targets;
5. preserve earlier interactions as training history;
6. keep non-positive interactions available for later modelling experiments;
7. verify that no target information leaks into training;
8. save processed tables for later model notebooks.

The raw source files will not be modified. All derived datasets will be written to `data/processed/`, which is excluded from Git because the files can be recreated by running this notebook.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent


raw_data_directory = (
    project_root
    / "data"
    / "raw"
)

processed_data_directory = (
    project_root
    / "data"
    / "processed"
)

processed_data_directory.mkdir(
    parents=True,
    exist_ok=True,
)


interaction_path = (
    raw_data_directory
    / "Video_Games.csv.gz"
)

metadata_path = (
    raw_data_directory
    / "meta_Video_Games.jsonl.gz"
)


interactions = pd.read_csv(
    interaction_path
)

interactions["interaction_time"] = pd.to_datetime(
    interactions["timestamp"],
    unit="ms",
    utc=True,
)


interactions = (
    interactions
    .sort_values(
        [
            "user_id",
            "interaction_time",
            "parent_asin",
        ]
    )
    .reset_index(drop=True)
)


print(
    "Project root:",
    project_root,
)

print(
    "Processed-data directory:",
    processed_data_directory,
)

print(
    "Interaction-data shape:",
    interactions.shape,
)

print(
    "Unique users:",
    interactions["user_id"].nunique(),
)

print(
    "Unique products:",
    interactions["parent_asin"].nunique(),
)

print(
    "Earliest interaction:",
    interactions["interaction_time"].min(),
)

print(
    "Latest interaction:",
    interactions["interaction_time"].max(),
)

interactions.head()

Project root: /Users/trevaogwang/Documents/mobile-projects/saidia_recommeder
Processed-data directory: /Users/trevaogwang/Documents/mobile-projects/saidia_recommeder/data/processed
Interaction-data shape: (814586, 5)
Unique users: 94762
Unique products: 25612
Earliest interaction: 1999-10-22 21:43:15+00:00
Latest interaction: 2023-09-02 18:14:28.630000+00:00


,user_id,parent_asin,rating,timestamp,interaction_time
0,AE222HFZDH6BPTYFOUWGGU63YSIQ,B082R1RGZF,5.0,1563320217391,2019-07-16 23:36:57.391000+00:00
1,AE222HFZDH6BPTYFOUWGGU63YSIQ,B07SNN8GV5,5.0,1576420340876,2019-12-15 14:32:20.876000+00:00
2,AE222HFZDH6BPTYFOUWGGU63YSIQ,B01GY35T4S,5.0,1578161337203,2020-01-04 18:08:57.203000+00:00
3,AE222HFZDH6BPTYFOUWGGU63YSIQ,B07QX99XJJ,5.0,1583088321887,2020-03-01 18:45:21.887000+00:00
4,AE222HFZDH6BPTYFOUWGGU63YSIQ,B0BW17W9GM,5.0,1593366227132,2020-06-28 17:43:47.132000+00:00


In [3]:
interactions["interaction_id"] = np.arange(
    len(interactions)
)

interactions["is_positive"] = (
    interactions["rating"] >= 4
)


positive_interactions = (
    interactions[
        interactions["is_positive"]
    ]
    .copy()
)

nonpositive_interactions = (
    interactions[
        ~interactions["is_positive"]
    ]
    .copy()
)


print(
    "All interactions:",
    f"{len(interactions):,}",
)

print(
    "Positive interactions:",
    f"{len(positive_interactions):,}",
)

print(
    "Non-positive interactions:",
    f"{len(nonpositive_interactions):,}",
)

print(
    "Recombined row count:",
    f"{len(positive_interactions) + len(nonpositive_interactions):,}",
)

print(
    "Users with at least one positive interaction:",
    f"{positive_interactions['user_id'].nunique():,}",
)

print(
    "Positive-rating values:",
    sorted(
        positive_interactions["rating"].unique()
    ),
)

print(
    "Non-positive rating values:",
    sorted(
        nonpositive_interactions["rating"].unique()
    ),
)

All interactions: 814,586
Positive interactions: 654,867
Non-positive interactions: 159,719
Recombined row count: 814,586
Users with at least one positive interaction: 93,949
Positive-rating values: [np.float64(4.0), np.float64(5.0)]
Non-positive rating values: [np.float64(1.0), np.float64(2.0), np.float64(3.0)]


In [4]:
positive_counts_per_user = (
    positive_interactions
    .groupby("user_id")
    .size()
    .rename("positive_interaction_count")
)

distinct_positive_times_per_user = (
    positive_interactions
    .groupby("user_id")["timestamp"]
    .nunique()
    .rename("distinct_positive_time_count")
)


user_evaluation_eligibility = pd.concat(
    [
        positive_counts_per_user,
        distinct_positive_times_per_user,
    ],
    axis=1,
)


user_evaluation_eligibility[
    "has_three_positive_interactions"
] = (
    user_evaluation_eligibility[
        "positive_interaction_count"
    ]
    >= 3
)

user_evaluation_eligibility[
    "has_three_distinct_positive_times"
] = (
    user_evaluation_eligibility[
        "distinct_positive_time_count"
    ]
    >= 3
)

user_evaluation_eligibility[
    "initially_eligible"
] = (
    user_evaluation_eligibility[
        "has_three_positive_interactions"
    ]
    & user_evaluation_eligibility[
        "has_three_distinct_positive_times"
    ]
)


initially_eligible_user_ids = set(
    user_evaluation_eligibility[
        user_evaluation_eligibility[
            "initially_eligible"
        ]
    ].index
)


print(
    "Users with at least one positive interaction:",
    f"{len(user_evaluation_eligibility):,}",
)

print(
    "Users with at least 3 positive interactions:",
    f"{user_evaluation_eligibility['has_three_positive_interactions'].sum():,}",
)

print(
    "Users with at least 3 distinct positive times:",
    f"{user_evaluation_eligibility['has_three_distinct_positive_times'].sum():,}",
)

print(
    "Initially eligible evaluation users:",
    f"{len(initially_eligible_user_ids):,}",
)

print(
    "Users with positive feedback but insufficient evaluation history:",
    f"{len(user_evaluation_eligibility) - len(initially_eligible_user_ids):,}",
)

display(
    user_evaluation_eligibility.head()
)

Users with at least one positive interaction: 93,949
Users with at least 3 positive interactions: 88,137
Users with at least 3 distinct positive times: 88,137
Initially eligible evaluation users: 88,137
Users with positive feedback but insufficient evaluation history: 5,812


,positive_interaction_count,distinct_positive_time_count,has_three_positive_interactions,has_three_distinct_positive_times,initially_eligible
user_id,,,,,
AE222HFZDH6BPTYFOUWGGU63YSIQ,5,5,True,True,True
AE2252DKW4XJIZP5QPFMQVJBVRTA,8,8,True,True,True
AE225O22SA7DLBOGOEIFL7FT5VYQ,7,7,True,True,True
AE227CCN4C37WTOB3J2TZPOKLEQQ,6,6,True,True,True
AE22BPPZGGRTSYOHK2J3LCG5HGAQ,5,5,True,True,True


In [5]:
eligible_positive_interactions = (
    positive_interactions[
        positive_interactions["user_id"].isin(
            initially_eligible_user_ids
        )
    ]
    .copy()
)


positive_timestamp_groups = (
    eligible_positive_interactions
    .groupby(
        ["user_id", "timestamp"]
    )
    .size()
    .rename("interaction_count")
    .reset_index()
    .sort_values(
        ["user_id", "timestamp"],
        ascending=[True, False],
    )
)


positive_timestamp_groups[
    "positive_time_position"
] = (
    positive_timestamp_groups
    .groupby("user_id")
    .cumcount()
    + 1
)


ambiguous_test_times = (
    positive_timestamp_groups[
        (
            positive_timestamp_groups[
                "positive_time_position"
            ]
            == 1
        )
        & (
            positive_timestamp_groups[
                "interaction_count"
            ]
            > 1
        )
    ]
)


ambiguous_validation_times = (
    positive_timestamp_groups[
        (
            positive_timestamp_groups[
                "positive_time_position"
            ]
            == 2
        )
        & (
            positive_timestamp_groups[
                "interaction_count"
            ]
            > 1
        )
    ]
)


ambiguous_test_user_ids = set(
    ambiguous_test_times["user_id"]
)

ambiguous_validation_user_ids = set(
    ambiguous_validation_times["user_id"]
)

ambiguous_target_user_ids = (
    ambiguous_test_user_ids
    | ambiguous_validation_user_ids
)

primary_evaluation_user_ids = (
    initially_eligible_user_ids
    - ambiguous_target_user_ids
)


print(
    "Initially eligible users:",
    f"{len(initially_eligible_user_ids):,}",
)

print(
    "Users with ambiguous test targets:",
    f"{len(ambiguous_test_user_ids):,}",
)

print(
    "Users with ambiguous validation targets:",
    f"{len(ambiguous_validation_user_ids):,}",
)

print(
    "Unique users with either ambiguity:",
    f"{len(ambiguous_target_user_ids):,}",
)

print(
    "Primary evaluation users:",
    f"{len(primary_evaluation_user_ids):,}",
)

display(
    positive_timestamp_groups.head(10)
)

Initially eligible users: 88,137
Users with ambiguous test targets: 5
Users with ambiguous validation targets: 8
Unique users with either ambiguity: 13
Primary evaluation users: 88,124


,user_id,timestamp,interaction_count,positive_time_position
4,AE222HFZDH6BPTYFOUWGGU63YSIQ,1593366227132,1,1
3,AE222HFZDH6BPTYFOUWGGU63YSIQ,1583088321887,1,2
2,AE222HFZDH6BPTYFOUWGGU63YSIQ,1578161337203,1,3
1,AE222HFZDH6BPTYFOUWGGU63YSIQ,1576420340876,1,4
0,AE222HFZDH6BPTYFOUWGGU63YSIQ,1563320217391,1,5
12,AE2252DKW4XJIZP5QPFMQVJBVRTA,1658440814568,1,1
11,AE2252DKW4XJIZP5QPFMQVJBVRTA,1562210954523,1,2
10,AE2252DKW4XJIZP5QPFMQVJBVRTA,1548397871869,1,3
9,AE2252DKW4XJIZP5QPFMQVJBVRTA,1532587156340,1,4
8,AE2252DKW4XJIZP5QPFMQVJBVRTA,1408995248000,1,5


In [6]:
timestamp_group_display = (
    positive_timestamp_groups
    .assign(
        interaction_time=pd.to_datetime(
            positive_timestamp_groups["timestamp"],
            unit="ms",
            utc=True,
        )
    )
    [
        [
            "user_id",
            "interaction_time",
            "interaction_count",
            "positive_time_position",
        ]
    ]
)

display(
    timestamp_group_display.head(10)
)

,user_id,interaction_time,interaction_count,positive_time_position
4,AE222HFZDH6BPTYFOUWGGU63YSIQ,2020-06-28 17:43:47.132000+00:00,1,1
3,AE222HFZDH6BPTYFOUWGGU63YSIQ,2020-03-01 18:45:21.887000+00:00,1,2
2,AE222HFZDH6BPTYFOUWGGU63YSIQ,2020-01-04 18:08:57.203000+00:00,1,3
1,AE222HFZDH6BPTYFOUWGGU63YSIQ,2019-12-15 14:32:20.876000+00:00,1,4
0,AE222HFZDH6BPTYFOUWGGU63YSIQ,2019-07-16 23:36:57.391000+00:00,1,5
12,AE2252DKW4XJIZP5QPFMQVJBVRTA,2022-07-21 22:00:14.568000+00:00,1,1
11,AE2252DKW4XJIZP5QPFMQVJBVRTA,2019-07-04 03:29:14.523000+00:00,1,2
10,AE2252DKW4XJIZP5QPFMQVJBVRTA,2019-01-25 06:31:11.869000+00:00,1,3
9,AE2252DKW4XJIZP5QPFMQVJBVRTA,2018-07-26 06:39:16.340000+00:00,1,4
8,AE2252DKW4XJIZP5QPFMQVJBVRTA,2014-08-25 19:34:08+00:00,1,5


## Positive-time position definition

`positive_time_position` is a derived value created from each user's positive-interaction timestamps. It is not a field supplied by the original Amazon dataset.

For every eligible user:

1. positive interactions are grouped by their recorded timestamp;
2. the distinct timestamps are sorted from newest to oldest;
3. the newest positive timestamp receives position 1;
4. the second-newest positive timestamp receives position 2;
5. earlier positive timestamps receive positions 3, 4 and so on.

Therefore, within one user:

- position 1 is the latest positive-interaction time;
- position 2 is the second-latest positive-interaction time;
- positions 3 and above represent earlier positive history.

The primary evaluation uses:

- position 1 as the test-target time;
- position 2 as the validation-target time;
- positions 3 and above as training-history times.

The position is calculated separately for every user. Position 1 for two different users does not necessarily represent the same calendar date.

If several positive products share the same recorded timestamp, they belong to the same positive-time position. The `interaction_count` column records how many positive products belong to that timestamp group.

Multiple products at position 1 create an ambiguous test target. Multiple products at position 2 create an ambiguous validation target.

The ordinary pandas row index has no chronological meaning. Chronological interpretation must use `interaction_time` and `positive_time_position`.

In [7]:
primary_target_time_groups = (
    positive_timestamp_groups[
        positive_timestamp_groups["user_id"].isin(
            primary_evaluation_user_ids
        )
        & positive_timestamp_groups[
            "positive_time_position"
        ].isin([1, 2])
    ]
    .copy()
)


test_time_keys = (
    primary_target_time_groups[
        primary_target_time_groups[
            "positive_time_position"
        ]
        == 1
    ]
    [
        [
            "user_id",
            "timestamp",
        ]
    ]
    .copy()
)


validation_time_keys = (
    primary_target_time_groups[
        primary_target_time_groups[
            "positive_time_position"
        ]
        == 2
    ]
    [
        [
            "user_id",
            "timestamp",
        ]
    ]
    .copy()
)


print(
    "Test-time keys:",
    f"{len(test_time_keys):,}",
)

print(
    "Validation-time keys:",
    f"{len(validation_time_keys):,}",
)

print(
    "Unique test-key users:",
    f"{test_time_keys['user_id'].nunique():,}",
)

print(
    "Unique validation-key users:",
    f"{validation_time_keys['user_id'].nunique():,}",
)

Test-time keys: 88,124
Validation-time keys: 88,124
Unique test-key users: 88,124
Unique validation-key users: 88,124


In [8]:
test_targets = (
    positive_interactions
    .merge(
        test_time_keys,
        on=[
            "user_id",
            "timestamp",
        ],
        how="inner",
        validate="many_to_one",
    )
    .sort_values("user_id")
    .reset_index(drop=True)
)

validation_targets = (
    positive_interactions
    .merge(
        validation_time_keys,
        on=[
            "user_id",
            "timestamp",
        ],
        how="inner",
        validate="many_to_one",
    )
    .sort_values("user_id")
    .reset_index(drop=True)
)

test_targets["target_type"] = "test"
validation_targets["target_type"] = "validation"

assert len(test_targets) == len(
    primary_evaluation_user_ids
)

assert len(validation_targets) == len(
    primary_evaluation_user_ids
)

assert test_targets["user_id"].is_unique
assert validation_targets["user_id"].is_unique

assert not (
    set(validation_targets["interaction_id"])
    & set(test_targets["interaction_id"])
)

print(
    "Primary evaluation users:",
    f"{len(primary_evaluation_user_ids):,}",
)

print(
    "Validation-target rows:",
    f"{len(validation_targets):,}",
)

print(
    "Unique validation-target users:",
    f"{validation_targets['user_id'].nunique():,}",
)

print(
    "Test-target rows:",
    f"{len(test_targets):,}",
)

print(
    "Unique test-target users:",
    f"{test_targets['user_id'].nunique():,}",
)

print(
    "Shared interaction IDs between target tables:",
    len(
        set(validation_targets["interaction_id"])
        & set(test_targets["interaction_id"])
    ),
)

Primary evaluation users: 88,124
Validation-target rows: 88,124
Unique validation-target users: 88,124
Test-target rows: 88,124
Unique test-target users: 88,124
Shared interaction IDs between target tables: 0


## Development and final-training configurations

Two positive-interaction training tables are required.

### Development training

During model development, both the validation and test targets remain hidden.

The model is trained using `development_positive_training`, evaluated against the validation targets and compared with alternative models or settings.

The test targets must not be used for model selection.

### Final training

After the model and settings have been selected, the validation targets may be returned to the training data.

The selected model is then trained using `final_positive_training`, while the test targets remain hidden for the final evaluation.

This creates the following flow:

1. development training → validation evaluation;
2. choose the model and settings;
3. final training, including validation interactions;
4. one final evaluation against the untouched test targets.

In [9]:
validation_target_ids = set(
    validation_targets["interaction_id"]
)

test_target_ids = set(
    test_targets["interaction_id"]
)

all_held_out_target_ids = (
    validation_target_ids
    | test_target_ids
)


development_positive_training = (
    positive_interactions[
        ~positive_interactions[
            "interaction_id"
        ].isin(all_held_out_target_ids)
    ]
    .copy()
)


final_positive_training = (
    positive_interactions[
        ~positive_interactions[
            "interaction_id"
        ].isin(test_target_ids)
    ]
    .copy()
)


print(
    "All positive interactions:",
    f"{len(positive_interactions):,}",
)

print(
    "Development positive-training rows:",
    f"{len(development_positive_training):,}",
)

print(
    "Validation-target rows:",
    f"{len(validation_targets):,}",
)

print(
    "Test-target rows:",
    f"{len(test_targets):,}",
)

print(
    "Development reconciliation total:",
    f"{len(development_positive_training) + len(validation_targets) + len(test_targets):,}",
)

print(
    "Final positive-training rows:",
    f"{len(final_positive_training):,}",
)

print(
    "Final reconciliation total:",
    f"{len(final_positive_training) + len(test_targets):,}",
)

All positive interactions: 654,867
Development positive-training rows: 478,619
Validation-target rows: 88,124
Test-target rows: 88,124
Development reconciliation total: 654,867
Final positive-training rows: 566,743
Final reconciliation total: 654,867


### Interpretation: development and final-training tables

The project uses two training configurations because validation and test have different purposes.

During model development:

- `development_positive_training` contains the positive interactions available for fitting candidate models;
- `validation_targets` contains one hidden positive interaction per primary evaluation user and is used to compare models and settings;
- `test_targets` contains one later hidden positive interaction per user and must not be used for model selection.

Both validation and test targets are therefore excluded from development training.

After the model and its settings have been selected, the validation interactions may be returned to the training data. The selected model can then be fitted using `final_positive_training`, while the test interactions remain hidden for the final evaluation.

The row counts reconcile with the original positive-interaction table:

- development training + validation targets + test targets = all positive interactions;
- final training + test targets = all positive interactions.

These reconciliation checks show that the split has not lost or duplicated positive-interaction rows. They do not, by themselves, prove that the split is chronologically correct. Chronological leakage checks are performed next.

In [10]:
test_targets.head(5)

,user_id,parent_asin,rating,timestamp,interaction_time,interaction_id,is_positive,target_type
0,AE222HFZDH6BPTYFOUWGGU63YSIQ,B0BW17W9GM,5.0,1593366227132,2020-06-28 17:43:47.132000+00:00,4,True,test
1,AE2252DKW4XJIZP5QPFMQVJBVRTA,B09DH4J73J,5.0,1658440814568,2022-07-21 22:00:14.568000+00:00,15,True,test
2,AE225O22SA7DLBOGOEIFL7FT5VYQ,B0053BCML6,4.0,1370812046000,2013-06-09 21:07:26+00:00,24,True,test
3,AE227CCN4C37WTOB3J2TZPOKLEQQ,B000IGE78M,5.0,1442558171000,2015-09-18 06:36:11+00:00,34,True,test
4,AE22BPPZGGRTSYOHK2J3LCG5HGAQ,B0053BCML6,5.0,1418058672000,2014-12-08 17:11:12+00:00,40,True,test


In [11]:
validation_targets.head(5)

,user_id,parent_asin,rating,timestamp,interaction_time,interaction_id,is_positive,target_type
0,AE222HFZDH6BPTYFOUWGGU63YSIQ,B07QX99XJJ,5.0,1583088321887,2020-03-01 18:45:21.887000+00:00,3,True,validation
1,AE2252DKW4XJIZP5QPFMQVJBVRTA,B07JH3LSHN,5.0,1562210954523,2019-07-04 03:29:14.523000+00:00,13,True,validation
2,AE225O22SA7DLBOGOEIFL7FT5VYQ,B00136MBHA,5.0,1218652402000,2008-08-13 18:33:22+00:00,22,True,validation
3,AE227CCN4C37WTOB3J2TZPOKLEQQ,B000WCCURW,5.0,1328427538000,2012-02-05 07:38:58+00:00,31,True,validation
4,AE22BPPZGGRTSYOHK2J3LCG5HGAQ,B00KVOVBGM,5.0,1414326075000,2014-10-26 12:21:15+00:00,39,True,validation


## Chronological leakage checks

A chronological recommender must learn from the past and predict a later interaction.

For every primary evaluation user:

- all development positive-training interactions must occur before that user's validation target;
- all final positive-training interactions must occur before that user's test target.

If training contains an interaction at or after the target time, the model could receive information that would not have been available when the recommendation was supposedly made. This is called **temporal data leakage**.

The following checks compare each user's latest available training time with that user's hidden target time.

In [12]:
development_latest_times = (
    development_positive_training[
        development_positive_training["user_id"].isin(
            primary_evaluation_user_ids
        )
    ]
    .groupby(
        "user_id",
        as_index=False,
    )["interaction_time"]
    .max()
    .rename(
        columns={
            "interaction_time": "latest_development_time"
        }
    )
)

validation_target_times = (
    validation_targets[
        [
            "user_id",
            "interaction_time",
        ]
    ]
    .rename(
        columns={
            "interaction_time": "validation_target_time"
        }
    )
)

development_time_check = (
    development_latest_times
    .merge(
        validation_target_times,
        on="user_id",
        how="inner",
        validate="one_to_one",
    )
)

development_time_check[
    "has_development_leakage"
] = (
    development_time_check[
        "latest_development_time"
    ]
    >= development_time_check[
        "validation_target_time"
    ]
)

final_latest_times = (
    final_positive_training[
        final_positive_training["user_id"].isin(
            primary_evaluation_user_ids
        )
    ]
    .groupby(
        "user_id",
        as_index=False,
    )["interaction_time"]
    .max()
    .rename(
        columns={
            "interaction_time": "latest_final_training_time"
        }
    )
)

test_target_times = (
    test_targets[
        [
            "user_id",
            "interaction_time",
        ]
    ]
    .rename(
        columns={
            "interaction_time": "test_target_time"
        }
    )
)

final_time_check = (
    final_latest_times
    .merge(
        test_target_times,
        on="user_id",
        how="inner",
        validate="one_to_one",
    )
)

final_time_check[
    "has_final_leakage"
] = (
    final_time_check[
        "latest_final_training_time"
    ]
    >= final_time_check[
        "test_target_time"
    ]
)

development_leakage_count = (
    development_time_check[
        "has_development_leakage"
    ].sum()
)

final_leakage_count = (
    final_time_check[
        "has_final_leakage"
    ].sum()
)

assert len(development_time_check) == len(
    primary_evaluation_user_ids
)

assert len(final_time_check) == len(
    primary_evaluation_user_ids
)

assert development_leakage_count == 0
assert final_leakage_count == 0

print(
    "Primary evaluation users:",
    f"{len(primary_evaluation_user_ids):,}",
)

print(
    "Users checked in development configuration:",
    f"{len(development_time_check):,}",
)

print(
    "Development temporal-leakage violations:",
    f"{development_leakage_count:,}",
)

print(
    "Users checked in final configuration:",
    f"{len(final_time_check):,}",
)

print(
    "Final temporal-leakage violations:",
    f"{final_leakage_count:,}",
)

display(
    development_time_check.head()
)

display(
    final_time_check.head()
)

Primary evaluation users: 88,124
Users checked in development configuration: 88,124
Development temporal-leakage violations: 0
Users checked in final configuration: 88,124
Final temporal-leakage violations: 0


,user_id,latest_development_time,validation_target_time,has_development_leakage
0,AE222HFZDH6BPTYFOUWGGU63YSIQ,2020-01-04 18:08:57.203000+00:00,2020-03-01 18:45:21.887000+00:00,False
1,AE2252DKW4XJIZP5QPFMQVJBVRTA,2019-01-25 06:31:11.869000+00:00,2019-07-04 03:29:14.523000+00:00,False
2,AE225O22SA7DLBOGOEIFL7FT5VYQ,2007-04-13 21:47:03+00:00,2008-08-13 18:33:22+00:00,False
3,AE227CCN4C37WTOB3J2TZPOKLEQQ,2011-12-29 02:01:42+00:00,2012-02-05 07:38:58+00:00,False
4,AE22BPPZGGRTSYOHK2J3LCG5HGAQ,2014-10-26 12:20:59+00:00,2014-10-26 12:21:15+00:00,False


,user_id,latest_final_training_time,test_target_time,has_final_leakage
0,AE222HFZDH6BPTYFOUWGGU63YSIQ,2020-03-01 18:45:21.887000+00:00,2020-06-28 17:43:47.132000+00:00,False
1,AE2252DKW4XJIZP5QPFMQVJBVRTA,2019-07-04 03:29:14.523000+00:00,2022-07-21 22:00:14.568000+00:00,False
2,AE225O22SA7DLBOGOEIFL7FT5VYQ,2008-08-13 18:33:22+00:00,2013-06-09 21:07:26+00:00,False
3,AE227CCN4C37WTOB3J2TZPOKLEQQ,2012-02-05 07:38:58+00:00,2015-09-18 06:36:11+00:00,False
4,AE22BPPZGGRTSYOHK2J3LCG5HGAQ,2014-10-26 12:21:15+00:00,2014-12-08 17:11:12+00:00,False


### Interpretation: chronological leakage checks

All 88,124 primary evaluation users were checked in both training configurations.

The development configuration produced zero temporal-leakage violations: every user's latest development-training interaction occurred before that user's validation target.

The final configuration also produced zero temporal-leakage violations: every user's latest final-training interaction occurred before that user's test target.

These results confirm that the chronological positive-interaction splits follow the intended order:

`earlier training history → validation target → test target`

The recommender can therefore be evaluated on later positive interactions without exposing those target interactions during the corresponding training stage.

In [13]:
tables_to_save = {
    "development_positive_training": development_positive_training,
    "final_positive_training": final_positive_training,
    "validation_targets": validation_targets,
    "test_targets": test_targets,
    "nonpositive_interactions": nonpositive_interactions,
    "user_evaluation_eligibility": (
        user_evaluation_eligibility.reset_index()
    ),
}

saved_file_rows = []

for table_name, table in tables_to_save.items():
    output_path = (
        processed_data_directory
        / f"{table_name}.csv.gz"
    )

    table.to_csv(
        output_path,
        index=False,
        compression="gzip",
    )

    saved_file_rows.append(
        {
            "table_name": table_name,
            "row_count": len(table),
            "output_path": str(output_path),
        }
    )

saved_files = pd.DataFrame(saved_file_rows)

assert len(saved_files) == len(tables_to_save)

for output_path in saved_files["output_path"]:
    assert Path(output_path).exists()

display(saved_files)

,table_name,row_count,output_path
0,development_positive_training,478619,/Users/trevaogwang/Documents/mobile-projects/s...
1,final_positive_training,566743,/Users/trevaogwang/Documents/mobile-projects/s...
2,validation_targets,88124,/Users/trevaogwang/Documents/mobile-projects/s...
3,test_targets,88124,/Users/trevaogwang/Documents/mobile-projects/s...
4,nonpositive_interactions,159719,/Users/trevaogwang/Documents/mobile-projects/s...
5,user_evaluation_eligibility,93949,/Users/trevaogwang/Documents/mobile-projects/s...


### Interpretation: saved modelling tables

Six reproducible modelling tables were saved successfully in `data/processed/`.

The development configuration contains 478,619 positive-training interactions, 88,124 validation targets, and 88,124 untouched test targets. These partitions reconcile to the original 654,867 positive interactions.

The final configuration contains 566,743 positive-training interactions and 88,124 untouched test targets. The additional 88,124 final-training rows are the validation interactions returned after model selection.

The eligibility table retains all 93,949 users with at least one positive interaction, including users who were not eligible for the primary chronological evaluation. The primary evaluation itself contains 88,124 users.

The 159,719 ratings from 1 to 3 are preserved separately as non-positive feedback. The initial recommender will use positive-only implicit feedback; incorporating non-positive events safely is reserved for a later modelling experiment.

All expected output files passed file-existence checks. This completes the verified positive-interaction data-preparation stage.